# The EYE/N3 certificate checker, in a notebook

`eye/check.py` is a plain Python module, so this notebook **imports it** and
calls its functions directly. Nothing here shells out to `run_tests.sh`; the same
cases are re-run in-process as Python.

What it shows:

1. an honest certificate is **accepted**;
2. a forged one is **refused**, naming the first bad step;
3. the two certificate *kinds* (`oo-cert` OWL-RL vs `oo-horn`);
4. the eight contract cases (positive, forged, wrong-order, wrong-direction,
   missing list member, `oo-refute` refusal), all as Python;
5. the optional **proven anchor** (Rocq), and a real false pass it catches;
6. the one thing that *is* a process boundary: the CLI exit-code contract.

**What an `accept` means here.** `check.py` is the untrusted driver; the trusted
part is `checker.n3`, run by EYE. It carries **no machine-checked theorem**
(`OOCert.certificate_sound` is about the Lean function and is not reproved here).
An accept means *the rules licensed every step*, not *it is entailed*. With the
Rocq anchor set — and only for `oo-horn` — the accept gains the theorem.

## 0 · Import the checker

In [1]:
import importlib.util, json, subprocess, sys, re, tempfile
from pathlib import Path

EYE = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
CHECK = EYE / "check.py"
O = EYE.parent
FIX = O / "tests" / "fixtures"
ROCQ = O / "rocq" / "build" / "oo-horn-rocq"
assert CHECK.exists(), f"no check.py at {CHECK}"

spec = importlib.util.spec_from_file_location("eyecheck", CHECK)
eye = importlib.util.module_from_spec(spec); spec.loader.exec_module(eye)
print("imported :", eye.__name__, "from", CHECK)
print("fixtures :", FIX)
print("anchor   :", ROCQ, "(built)" if ROCQ.exists() else "(not built)")

imported : eyecheck from /Users/jim/agent-zero/agent-zero-1/usr/projects/pup/open-ontologies/eye/check.py
fixtures : /Users/jim/agent-zero/agent-zero-1/usr/projects/pup/open-ontologies/tests/fixtures
anchor   : /Users/jim/agent-zero/agent-zero-1/usr/projects/pup/open-ontologies/rocq/build/oo-horn-rocq (built)


## 1 · Check a certificate, in process

`check.py` exposes exactly what `main()` uses: `parse_asserted`, `parse_steps`
(`oo-cert`), `parse_horn_steps` (`oo-horn`), and `check_steps(asserted, steps)
-> (ok, first_bad_index)`. The helper decides nothing itself.

In [2]:
def check_texts(asserted_text, deriv_text, fmt="oo-cert"):
    asserted = eye.parse_asserted(asserted_text)
    steps = eye.parse_steps(deriv_text) if fmt == "oo-cert" else eye.parse_horn_steps(deriv_text)
    ok, idx = eye.check_steps(asserted, steps)
    return ok, idx, steps

def show(label, asserted_path, cert_path):
    fmt, steps, _ = eye.load_derivations(Path(cert_path))
    asserted = eye.parse_asserted(Path(asserted_path).read_text())
    ok, idx = eye.check_steps(asserted, steps)
    verdict = "ACCEPT" if ok else f"REJECT at step {idx}"
    print(f"{label:<46} {fmt:<8} {verdict}")
    if not ok:
        rule, concl, _ = steps[idx]
        print("     rule:", rule, "| conclusion:", " ".join(concl))
    return fmt, ok

### 1a · An honest certificate is accepted

In [3]:
show("honest rdfs9 certificate", FIX / "refute/asserted.tsv", FIX / "refute/derivations.tsv")

honest rdfs9 certificate                       oo-cert  ACCEPT


('oo-cert', True)

### 1b · A forged one is refused

Same certificate, change **only the conclusion object** (`Carnivore` ->
`Herbivore`), both premises exactly as they were.

In [4]:
src = (FIX / "refute/derivations.tsv").read_text()
forged = re.sub("Carnivore", "Herbivore", src, count=1)
forged_path = EYE / "examples" / "forged_derivations.tsv"
forged_path.write_text(forged)
print("before:", src.splitlines()[0].split("\t")[:4])
print("after :", forged.splitlines()[0].split("\t")[:4]); print()
show("forged certificate", FIX / "refute/asserted.tsv", forged_path)

before: ['rdfs9', '<http://ex.org/leo>', '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>', '<http://ex.org/Carnivore>']
after : ['rdfs9', '<http://ex.org/leo>', '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>', '<http://ex.org/Herbivore>']

forged certificate                             oo-cert  REJECT at step 0
     rule: rdfs9 | conclusion: <http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Herbivore>


('oo-cert', False)

## 2 · Two certificate kinds (auto-detected)

- **`oo-cert`** — `derivations.tsv`: the OWL-RL rule set (29 arms).
- **`oo-horn`** — `horn.tsv`: the Horn rule set (the Rocq-proven one).

`load_derivations` picks the parser from the file's own shape.

In [5]:
show("oo-horn (Horn rule set)", FIX / "horn/asserted.tsv", FIX / "horn/good.tsv")

oo-horn (Horn rule set)                        oo-horn  ACCEPT


('oo-horn', True)

## 3 · The contract cases, in Python

This is `run_tests.sh`'s job done in the notebook: the same eight cases, the
negative fixtures built inline as text, every check run through the imported
module. No shell script, no subprocess.

The cases:

| case | what it proves |
|---|---|
| positive `oo-cert` | a real certificate is accepted |
| positive `oo-horn` | a Horn certificate is accepted |
| forged | one changed conclusion in a real certificate is refused |
| (a) premise asserted nowhere | an unbacked premise is refused |
| (b) premises in wrong order | order is part of the contract |
| (c) `scm-avf2` natural direction | the one reversed rule is enforced the right way round |
| (d) `cls-int1` missing a member type | every list member must be typed |
| `oo-refute/1` file | refused as a different kind of sentence, not mis-checked |

In [6]:
RDF="http://www.w3.org/1999/02/22-rdf-syntax-ns"; OWL="http://www.w3.org/2002/07/owl"
RDFS="http://www.w3.org/2000/01/rdf-schema"
T=f"<{RDF}#type>"; SC=f"<{RDFS}#subClassOf>"; SP=f"<{RDFS}#subPropertyOf>"
AVF=f"<{OWL}#allValuesFrom>"; OP=f"<{OWL}#onProperty>"; ISEC=f"<{OWL}#intersectionOf>"
FIRST=f"<{RDF}#first>"; REST=f"<{RDF}#rest>"; NIL=f"<{RDF}#nil>"

def row(*f): return "\t".join(f)

# (a) a premise asserted nowhere and concluded nowhere
A_a = row("<http://ex.org/B>", SC, "<http://ex.org/C>")
D_a = row("rdfs9", "<http://ex.org/a>", T, "<http://ex.org/C>",
          "<http://ex.org/a>", T, "<http://ex.org/B>", "<http://ex.org/B>", SC, "<http://ex.org/C>")

# (b) rdfs9 with the premises in the wrong order (both asserted)
A_b = row("<http://ex.org/a>", T, "<http://ex.org/A>") + "\n" + row("<http://ex.org/A>", SC, "<http://ex.org/B>")
D_b = row("rdfs9", "<http://ex.org/a>", T, "<http://ex.org/B>",
          "<http://ex.org/A>", SC, "<http://ex.org/B>", "<http://ex.org/a>", T, "<http://ex.org/A>")

# (c) scm-avf2 with the conclusion in the natural (wrong) direction c1 sc c2
A_c = "\n".join([row("<http://ex.org/C1>", AVF, "<http://ex.org/Y>"),
                  row("<http://ex.org/C1>", OP, "<http://ex.org/p1>"),
                  row("<http://ex.org/C2>", AVF, "<http://ex.org/Y>"),
                  row("<http://ex.org/C2>", OP, "<http://ex.org/p2>"),
                  row("<http://ex.org/p1>", SP, "<http://ex.org/p2>")])
D_c = row("scm-avf2", "<http://ex.org/C1>", SC, "<http://ex.org/C2>",
          "<http://ex.org/C1>", AVF, "<http://ex.org/Y>", "<http://ex.org/C1>", OP, "<http://ex.org/p1>",
          "<http://ex.org/C2>", AVF, "<http://ex.org/Y>", "<http://ex.org/C2>", OP, "<http://ex.org/p2>",
          "<http://ex.org/p1>", SP, "<http://ex.org/p2>")

# (d) cls-int1 with one member's type missing
A_d = "\n".join([row("<http://ex.org/c>", ISEC, "<http://ex.org/l>"),
                  row("<http://ex.org/l>", FIRST, "<http://ex.org/A>"),
                  row("<http://ex.org/l>", REST, "<http://ex.org/l2>"),
                  row("<http://ex.org/l2>", FIRST, "<http://ex.org/B>"),
                  row("<http://ex.org/l2>", REST, NIL),
                  row("<http://ex.org/x>", T, "<http://ex.org/A>")])
D_d = row("cls-int1", "<http://ex.org/x>", T, "<http://ex.org/c>",
          "<http://ex.org/c>", ISEC, "<http://ex.org/l>",
          "<http://ex.org/l>", FIRST, "<http://ex.org/A>",
          "<http://ex.org/l>", REST, "<http://ex.org/l2>",
          "<http://ex.org/l2>", FIRST, "<http://ex.org/B>",
          "<http://ex.org/l2>", REST, NIL,
          "<http://ex.org/x>", T, "<http://ex.org/A>")

ran=fails=0
def expect(want_ok, label, ok, idx=None):
    global ran, fails
    ran += 1
    good = (ok == want_ok)
    print(f"{'PASS' if good else 'FAIL'} [{label}] " + ("accept" if ok else f"reject at step {idx}"))
    if not good: fails += 1

def case(label, want_ok, asserted_path, cert_path):
    fmt, ok = show(label, asserted_path, cert_path)
    expect(want_ok, label, ok)

def case_text(label, want_ok, asserted_text, deriv_text, fmt="oo-cert"):
    ok, idx, _ = check_texts(asserted_text, deriv_text, fmt)
    expect(want_ok, label, ok, idx)

# positives and the forged real certificate
case("pos: oo-cert refute/derivations.tsv", True, FIX / "refute/asserted.tsv", FIX / "refute/derivations.tsv")
case("pos: oo-horn horn/good.tsv",        True, FIX / "horn/asserted.tsv",   FIX / "horn/good.tsv")
case("neg: forged conclusion",            False, FIX / "refute/asserted.tsv", forged_path)
case("neg: oo-horn forged",               False, FIX / "horn/asserted.tsv",   FIX / "horn/supplier/forged.tsv")
# contract cases (in-process)
case_text("neg(a): unasserted/underived premise", False, A_a, D_a)
case_text("neg(b): premises in wrong order",       False, A_b, D_b)
case_text("neg(c): scm-avf2 natural direction",    False, A_c, D_c)
case_text("neg(d): cls-int1 missing member type",  False, A_d, D_d)
print(f"\nran {ran}, failed {fails}")
assert fails == 0, "contract cases failed"

pos: oo-cert refute/derivations.tsv            oo-cert  ACCEPT
PASS [pos: oo-cert refute/derivations.tsv] accept
pos: oo-horn horn/good.tsv                     oo-horn  ACCEPT
PASS [pos: oo-horn horn/good.tsv] accept
neg: forged conclusion                         oo-cert  REJECT at step 0
     rule: rdfs9 | conclusion: <http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Herbivore>
PASS [neg: forged conclusion] reject at step None
neg: oo-horn forged                            oo-horn  REJECT at step 0
     rule: rdfs9 | conclusion: <http://ex.org/Northwind> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/HighRiskSupplier>
PASS [neg: oo-horn forged] reject at step None
PASS [neg(a): unasserted/underived premise] reject at step 0
PASS [neg(b): premises in wrong order] reject at step 0
PASS [neg(c): scm-avf2 natural direction] reject at step 0
PASS [neg(d): cls-int1 missing member type] reject at step 0

ran 8, failed 0


### 3a · `oo-refute/1` is a different sentence

A refutation file starts with the tag `oo-refute/1`. It is **not** an `oo-cert`
certificate, so it earns a different verdict: `load_derivations` reports the
format and the driver refuses to check it as one. (The CLI exits `2`.)

In [7]:
fmt, steps, refute_line = eye.load_derivations(FIX / "refute/good.tsv")
print("format    :", fmt)
print("step count:", len(steps))
print("refute    :", refute_line)
assert fmt == "oo-refute", "expected oo-refute/1"
print("→ this is a refutation, not a derivation; oo-cert does not check it")

format    : oo-refute
step count: 1
refute    : refute	cax-dw	<http://ex.org/Herbivore>	<http://www.w3.org/2002/07/owl#disjointWith>	<http://ex.org/Carnivore>	<http://ex.org/leo>	<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>	<http://ex.org/Herbivore>	<http://ex.org/leo>	<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>	<http://ex.org/Carnivore>
→ this is a refutation, not a derivation; oo-cert does not check it


## 4 · The optional proven anchor (Rocq)

`check.py` runs the **Rocq-extracted, machine-checked** `oo-horn` checker
(theorem `entails_of_builtin_horn`) over the same inputs when its `ANCHOR` and
`RULES_TSV` are set. We set them on the module directly (the CLI sets them from
`OO_HORN_ROCQ`/`OO_RULES`).

- agree -> the accept carries the theorem name;
- disagree on an `oo-horn` certificate -> the CLI exits **3**, not 0;
- on an `oo-cert` certificate the anchor **declines** (Horn-only).

In [8]:
def anchor(asserted_path, cert_path, binary=ROCQ, rules=None):
    eye.ANCHOR = str(binary)
    eye.RULES_TSV = str(rules if rules is not None else FIX / "horn/builtin_rules.tsv")
    try:
        return eye.run_anchor(str(cert_path), str(asserted_path))
    except RuntimeError as e:
        return {"exit": None, "error": str(e)}
    finally:
        eye.ANCHOR = None; eye.RULES_TSV = None

if ROCQ.exists():
    good = anchor(FIX / "horn/asserted.tsv", FIX / "horn/good.tsv")
    if good.get("exit") is None:
        print("anchor could not run:", good["error"])
        print("  (rebuild: (cd", O / "rocq", "&& bash build.sh))")
    else:
        print("oo-horn good ->", good["exit"], "|", good["verdict"].get("theorem"))
        bad = anchor(FIX / "horn/asserted.tsv", FIX / "horn/bad_conclusion.tsv")
        print("oo-horn bad  ->", bad["exit"], "| EYE ACCEPT, anchor REJECT => CAUGHT FALSE PASS" if bad["exit"] != 0 else "")
        print("oo-cert      ->", json.dumps(eye.anchor_declined("oo-cert")))
else:
    print("The Rocq anchor is not built here. Build it with:")
    print("  (cd", O / "rocq", "&& bash build.sh)")

oo-horn good -> 0 | OOCertRocq.entails_of_builtin_horn
oo-horn bad  -> 1 | EYE ACCEPT, anchor REJECT => CAUGHT FALSE PASS
oo-cert      -> {"applied": false, "reason": "oo-horn (Horn) checker only; oo-cert uses the OWL-RL rule set (incl. the RDF-list rules it does not cover)"}


### Why `bad_conclusion` matters

The fixture has a binding `b = Z` contradicting its own conclusion object `B`.
EYE **accepts** it (the Horn bindings are dropped when the rule index is mapped
through `asHorn`); the proven checker **refuses** it. A textbook **false pass** —
surfaced only by running the two together. The CLI turns it into exit **3**.

## 5 · The one process boundary: the CLI exit-code contract

Every check above ran in-process. A pipeline (and the *Try It* page) does not
call Python functions — it runs `check.py` and reads an **exit code**: `0` accept,
`1` reject, `2` parse error, `3` **only with the anchor** when EYE accepts but
the theorem refuses. That is a property of the process, not of the functions, so
this cell — the only `subprocess` in the notebook — shows it.

In [9]:
import os
E = dict(os.environ); E["OO_HORN_ROCQ"]=str(ROCQ); E["OO_RULES"]=str(FIX / "horn/builtin_rules.tsv")
def cli(label, asserted_path, cert_path, env=E):
    p = subprocess.run([sys.executable, str(CHECK), str(asserted_path), str(cert_path)],
                       capture_output=True, text=True, cwd=str(EYE), env=env)
    line = next((l for l in p.stdout.splitlines() if l.startswith("{")), "")
    print(f"{label:<42} exit={p.returncode}")
    print("   ", line[:240] if line else p.stderr[-200:])
cli("CLI: honest oo-cert", FIX / "refute/asserted.tsv", FIX / "refute/derivations.tsv")
cli("CLI: forged oo-cert", FIX / "refute/asserted.tsv", forged_path)
cli("CLI: oo-horn good + anchor", FIX / "horn/asserted.tsv", FIX / "horn/good.tsv")
cli("CLI: oo-horn bad_conclusion + anchor", FIX / "horn/asserted.tsv", FIX / "horn/bad_conclusion.tsv")

CLI: honest oo-cert                        exit=0
    {"ok": true, "asserted": 4, "derivations": 1, "anchor": {"applied": false, "reason": "oo-horn (Horn) checker only; oo-cert uses the OWL-RL rule set (incl. the RDF-list rules it does not cover)"}}
CLI: forged oo-cert                        exit=1
    {"ok": false, "rejected_step": 0, "rule": "rdfs9", "conclusion": "<http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Herbivore>", "premises": ["<http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-n
CLI: oo-horn good + anchor                 exit=0
    {"ok": true, "asserted": 2, "derivations": 1, "anchor": {"binary": "oo-horn-rocq", "exit": 0, "verdict": {"ok": true, "kernel": "rocq", "verdict": "entailed", "rules": 27, "asserted": 2, "derivations": 1, "theorem": "OOCertRocq.entails_of_b
CLI: oo-horn bad_conclusion + anchor       exit=3
    {"ok": false, "asserted": 2, "derivations": 1, "anchor": {"binary": "oo-horn-rocq", "exit": 1, "verdict": 

## 6 · What this does *not* claim

- The `oo-cert` soundness theorem (`OOCert.certificate_sound`) is about the
  **Lean** function and is **not** reproved here. For `oo-cert`, an accept is
  *the rules licensed every step*.
- The Rocq anchor proves a theorem about the **Horn** rule set only
  (`entails_of_builtin_horn`); it applies to `oo-horn`, not `oo-cert`.
- Even with the anchor, the verdict is **conditional on the asserted graph**:
  neither EYE nor Rocq checks the graph is the one the engine actually reasoned
  over.

That is the honest boundary of this notebook, and of `eye/` itself.